In [1]:
# Hand Gesture Recognition (ML)
# HOG Feature Extraction + SVM

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# STEP 1: Dataset Path & Settings
DATASET_PATH = "leapGestRecog"   # change path if needed
IMG_SIZE = 64

features = []
labels = []

In [ ]:
# STEP 2: Load Images & Extract HOG
print("Loading images and extracting features...")

for person in os.listdir(DATASET_PATH):
    person_path = os.path.join(DATASET_PATH, person)

    if not os.path.isdir(person_path):
        continue

    for gesture in os.listdir(person_path):
        gesture_path = os.path.join(person_path, gesture)

        if not os.path.isdir(gesture_path):
            continue

        for img_name in os.listdir(gesture_path):
            img_path = os.path.join(gesture_path, img_name)

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            # 🔴 VERY IMPORTANT CHECK
            if img is None:
                print(f"Skipping unreadable image: {img_path}")
                continue

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

            hog_features = hog(
                img,
                orientations=9,
                pixels_per_cell=(8, 8),
                cells_per_block=(2, 2),
                block_norm='L2-Hys'
            )

            features.append(hog_features)
            labels.append(gesture)


print("Feature extraction completed.")


Loading images and extracting features...


In [15]:
# STEP 3: Convert to NumPy & Encode Labels

X = np.array(features)
y = np.array(labels)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [16]:
# STEP 4: Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

In [17]:
# STEP 5: Train SVM Classifier

print("Training SVM model...")

model = SVC(kernel='rbf', C=10, gamma='scale')
model.fit(X_train, y_train)

print("Model training completed.")

Training SVM model...
Model training completed.


In [18]:
# STEP 6: Evaluate Model

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("\nModel Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


Model Accuracy: 1.0

Classification Report:

               precision    recall  f1-score   support

      01_palm       1.00      1.00      1.00       417
         02_l       1.00      1.00      1.00       404
      03_fist       1.00      1.00      1.00       404
04_fist_moved       1.00      1.00      1.00       418
     05_thumb       1.00      1.00      1.00       377
     06_index       1.00      1.00      1.00       392
        07_ok       1.00      1.00      1.00       403
08_palm_moved       1.00      1.00      1.00       409
         09_c       1.00      1.00      1.00       410
      10_down       1.00      1.00      1.00       366

     accuracy                           1.00      4000
    macro avg       1.00      1.00      1.00      4000
 weighted avg       1.00      1.00      1.00      4000



In [20]:
# STEP 7: Gesture Prediction Function

def predict_gesture(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    hog_features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )

    hog_features = hog_features.reshape(1, -1)
    prediction = model.predict(hog_features)

    return label_encoder.inverse_transform(prediction)[0]

In [21]:
test_image_path = "C:/Users/JANANI_ABOORVAVARSHA/Downloads/leapGestRecog/00/01_palm/frame_00_01_0001.png"
result = predict_gesture(test_image_path)
print("Predicted Hand Gesture:", result)


Predicted Hand Gesture: 01_palm
